In [ ]:
!pip -q install torch torchvision opencv-python pandas scikit-learn tqdm matplotlib

In [ ]:
import os, zipfile, shutil, math, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Updated to match the files actually present in /content
ZIP_NAMES = [
    "CV_Dataset.zip",
    "Lichen-20260608T222504Z-3-001.zip",
]

# Search both standard Colab /content and the chat runtime /mnt/data
ZIP_PATHS = []
for name in ZIP_NAMES:
    found = None
    for base in ["/content", "/mnt/data"]:
        p = Path(base) / name
        if p.exists():
            found = p
            break
    if found:
        ZIP_PATHS.append(found)
    else:
        print(f"WARNING: Could not find {name}. Please ensure it is uploaded.")

assert ZIP_PATHS, "No zip files found."

WORK_DIR = Path("/content/lichen_100_binary")
RAW_DIR = WORK_DIR / "raw_extracted"
DATASET_DIR = WORK_DIR / "dataset"
IMG_DIR = DATASET_DIR / "images"
MASK_DIR = DATASET_DIR / "masks"

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

RAW_DIR.mkdir(parents=True)
IMG_DIR.mkdir(parents=True)
MASK_DIR.mkdir(parents=True)

# Extract all zips.
for zp in ZIP_PATHS:
    out = RAW_DIR / zp.stem
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(out)
    print("Extracted:", zp.name)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def is_real_image_file(p: Path):
    s = p.stem.lower()
    if p.suffix.lower() not in IMAGE_EXTS:
        return False
    if "_mask" in s or "_overlay" in s or "_review" in s or "comparison" in s:
        return False
    return True

# Build mask lookup by cleaned stem.
all_files = list(RAW_DIR.rglob("*"))
mask_candidates = []
for p in all_files:
    if p.suffix.lower() in IMAGE_EXTS:
        s = p.stem.lower()
        if "_mask" in s:
            mask_candidates.append(p)

mask_lookup = {}
for mp in mask_candidates:
    stem = mp.stem
    clean_stem = stem.replace("_mask", "")
    mask_lookup.setdefault(clean_stem, []).append(mp)

records = []
used = set()
pair_id = 0

for img_p in sorted([p for p in all_files if is_real_image_file(p)]):
    stem = img_p.stem

    # Try exact mask match first.
    possible_masks = []
    possible_masks += list(img_p.parent.glob(f"{stem}_mask.png"))
    possible_masks += list(img_p.parent.glob(f"{stem}_mask.jpg"))
    possible_masks += list(img_p.parent.glob(f"{stem}.png"))
    possible_masks += mask_lookup.get(stem, [])

    possible_masks = [p for p in possible_masks if p.exists()]
    if not possible_masks:
        continue

    mask_p = possible_masks[0]

    # Avoid duplicates based on original stem + mask path.
    key = (stem, str(mask_p.name))
    if key in used:
        continue
    used.add(key)

    img = cv2.imread(str(img_p))
    mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        continue

    h, w = img.shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    # Force true binary mask.
    mask_bin = (mask > 127).astype(np.uint8) * 255
    coverage = float((mask_bin > 0).mean() * 100)

    # Skip broken masks.
    if coverage < 0.1 or coverage > 90:
        print("Skipping suspicious coverage:", img_p.name, coverage)
        continue

    pair_id += 1
    new_img_name = f"lichen_{pair_id:04d}.jpg"
    new_mask_name = f"lichen_{pair_id:04d}_mask.png"

    # Save normalized image/mask pair.
    cv2.imwrite(str(IMG_DIR / new_img_name), img)
    cv2.imwrite(str(MASK_DIR / new_mask_name), mask_bin)

    records.append({
        "image_id": f"lichen_{pair_id:04d}",
        "image_path": str(IMG_DIR / new_img_name),
        "mask_path": str(MASK_DIR / new_mask_name),
        "coverage_percent": coverage,
        "original_image": str(img_p),
        "original_mask": str(mask_p),
    })

df = pd.DataFrame(records)
print("Total valid image-mask pairs found:", len(df))

# Use around 100, but keep diversity by coverage bins.
MAX_IMAGES = 100

if len(df) > MAX_IMAGES:
    df["coverage_bin"] = pd.qcut(
        df["coverage_percent"],
        q=min(5, len(df)),
        labels=False,
        duplicates="drop"
    )

    sampled = []
    bins = sorted(df["coverage_bin"].dropna().unique())
    per_bin = math.ceil(MAX_IMAGES / max(1, len(bins)))

    for b in bins:
        part = df[df["coverage_bin"] == b]
        sampled.append(part.sample(min(len(part), per_bin), random_state=SEED))

    df = pd.concat(sampled).sample(frac=1, random_state=SEED).head(MAX_IMAGES).reset_index(drop=True)

df = df.drop(columns=["coverage_bin"], errors="ignore")
df.to_csv(DATASET_DIR / "labels_all.csv", index=False)

print("Using final dataset size:", len(df))
print("Saved:", DATASET_DIR / "labels_all.csv")
display(df.head())

In [ ]:
import os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import resnet34, ResNet34_Weights

from sklearn.model_selection import train_test_split

# -----------------------------
# Config
# -----------------------------

DATASET_DIR = Path("/content/lichen_100_binary/dataset")
LABELS_CSV = DATASET_DIR / "labels_all.csv"

RUN_DIR = Path("/content/runs/lichen_binary_resnet34_unet_100")
RUN_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 384
BATCH_SIZE = 4
EPOCHS = 60
FREEZE_ENCODER_EPOCHS = 5
PATIENCE = 12

LR_FROZEN = 1e-3
LR_UNFROZEN = 2e-4
WEIGHT_DECAY = 1e-4

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Dataset split
# -----------------------------

df = pd.read_csv(LABELS_CSV)

# Stratify by coverage so train/val both include small and large masks.
df["coverage_bin"] = pd.qcut(
    df["coverage_percent"],
    q=min(4, len(df)),
    labels=False,
    duplicates="drop"
)

try:
    train_df, val_df = train_test_split(
        df,
        test_size=0.20,
        random_state=SEED,
        stratify=df["coverage_bin"]
    )
except Exception:
    train_df, val_df = train_test_split(
        df,
        test_size=0.20,
        random_state=SEED
    )

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df.to_csv(RUN_DIR / "train_split.csv", index=False)
val_df.to_csv(RUN_DIR / "val_split.csv", index=False)

print("Train:", len(train_df), "Val:", len(val_df))

# -----------------------------
# Image/mask helpers
# -----------------------------

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def resize_pair(image, mask, size=384):
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)
    return image, mask

def augment_pair(image, mask):
    # Flips
    if random.random() < 0.5:
        image = cv2.flip(image, 1)
        mask = cv2.flip(mask, 1)
    if random.random() < 0.15:
        image = cv2.flip(image, 0)
        mask = cv2.flip(mask, 0)

    h, w = image.shape[:2]

    # Rotation + slight translation/scale
    if random.random() < 0.75:
        angle = random.uniform(-25, 25)
        scale = random.uniform(0.90, 1.10)
        tx = random.uniform(-0.05, 0.05) * w
        ty = random.uniform(-0.05, 0.05) * h

        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
        M[0, 2] += tx
        M[1, 2] += ty

        image = cv2.warpAffine(
            image, M, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT_101
        )
        mask = cv2.warpAffine(
            mask, M, (w, h),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0
        )

    # Brightness/contrast jitter
    if random.random() < 0.8:
        alpha = random.uniform(0.75, 1.25)  # contrast
        beta = random.uniform(-25, 25)      # brightness
        image = np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

    # Mild blur sometimes, because field photos are imperfect.
    if random.random() < 0.15:
        image = cv2.GaussianBlur(image, (3, 3), 0)

    # Mild HSV jitter
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 1] *= random.uniform(0.85, 1.20)
        hsv[:, :, 2] *= random.uniform(0.85, 1.15)
        hsv = np.clip(hsv, 0, 255).astype(np.uint8)
        image = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    return image, mask

class LichenMaskDataset(Dataset):
    def __init__(self, dataframe, img_size=384, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        bgr = cv2.imread(row["image_path"])
        if bgr is None:
            raise ValueError(f"Could not read image: {row['image_path']}")
        image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Could not read mask: {row['mask_path']}")

        mask = (mask > 127).astype(np.uint8)
        image, mask = resize_pair(image, mask, self.img_size)

        if self.augment:
            image, mask = augment_pair(image, mask)

        image = image.astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD

        image = np.transpose(image, (2, 0, 1)).astype(np.float32)
        mask = mask.astype(np.float32)[None, :, :]

        return torch.tensor(image), torch.tensor(mask)

train_ds = LichenMaskDataset(train_df, img_size=IMG_SIZE, augment=True)
val_ds = LichenMaskDataset(val_df, img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# -----------------------------
# ResNet34 U-Net model
# -----------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ResNet34UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        try:
            weights = ResNet34_Weights.DEFAULT if pretrained else None
            base = resnet34(weights=weights)
        except Exception as e:
            print("Could not load pretrained weights. Using random weights.", e)
            base = resnet34(weights=None)

        self.enc0 = nn.Sequential(base.conv1, base.bn1, base.relu)  # /2, 64
        self.pool = base.maxpool                                  # /4
        self.enc1 = base.layer1                                   # /4, 64
        self.enc2 = base.layer2                                   # /8, 128
        self.enc3 = base.layer3                                   # /16, 256
        self.enc4 = base.layer4                                   # /32, 512

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.up0 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec0 = DoubleConv(32, 32)

        self.final = nn.Conv2d(32, 1, kernel_size=1)

    def encoder_parameters(self):
        modules = [self.enc0, self.pool, self.enc1, self.enc2, self.enc3, self.enc4]
        for m in modules:
            for p in m.parameters():
                yield p

    def forward(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(self.pool(e0))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        d4 = self.up4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e0], dim=1)
        d1 = self.dec1(d1)

        d0 = self.up0(d1)
        d0 = self.dec0(d0)

        return self.final(d0)

model = ResNet34UNet(pretrained=True).to(device)

# -----------------------------
# Loss + metrics
# -----------------------------

class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        probs = torch.sigmoid(logits)

        smooth = 1e-6
        intersection = (probs * targets).sum(dim=(1, 2, 3))
        union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - ((2 * intersection + smooth) / (union + smooth))
        dice_loss = dice_loss.mean()

        return self.bce_weight * bce + self.dice_weight * dice_loss

def batch_dice_iou(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    smooth = 1e-6

    intersection = (preds * targets).sum(dim=(1, 2, 3))
    pred_sum = preds.sum(dim=(1, 2, 3))
    target_sum = targets.sum(dim=(1, 2, 3))

    dice = (2 * intersection + smooth) / (pred_sum + target_sum + smooth)
    iou = (intersection + smooth) / (pred_sum + target_sum - intersection + smooth)

    return dice.mean().item(), iou.mean().item()

criterion = DiceBCELoss(bce_weight=0.45, dice_weight=0.55)

def set_encoder_frozen(model, frozen=True):
    for p in model.encoder_parameters():
        p.requires_grad = not frozen

set_encoder_frozen(model, True)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FROZEN, weight_decay=WEIGHT_DECAY)

# -----------------------------
# Training loop
# -----------------------------

best_val_dice = -1
best_epoch = -1
no_improve = 0
log_rows = []

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_ENCODER_EPOCHS + 1:
        print("Unfreezing encoder...")
        set_encoder_frozen(model, False)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_UNFROZEN, weight_decay=WEIGHT_DECAY)

    model.train()
    train_loss = 0
    train_dice = 0
    train_iou = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} train", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        d, i = batch_dice_iou(logits.detach(), masks, threshold=0.5)
        train_loss += loss.item()
        train_dice += d
        train_iou += i

    train_loss /= len(train_loader)
    train_dice /= len(train_loader)
    train_iou /= len(train_loader)

    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} val", leave=False):
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            d, i = batch_dice_iou(logits, masks, threshold=0.5)
            val_loss += loss.item()
            val_dice += d
            val_iou += i

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    val_iou /= len(val_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_dice": train_dice,
        "train_iou": train_iou,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
    }
    log_rows.append(row)
    pd.DataFrame(log_rows).to_csv(RUN_DIR / "training_log.csv", index=False)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss {train_loss:.4f} Dice {train_dice:.4f} IoU {train_iou:.4f} | "
        f"Val Loss {val_loss:.4f} Dice {val_dice:.4f} IoU {val_iou:.4f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch = epoch
        no_improve = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "img_size": IMG_SIZE,
            "epoch": epoch,
            "val_dice_05": val_dice,
            "val_iou_05": val_iou,
            "threshold": 0.5,
        }, RUN_DIR / "best_model_raw_threshold.pt")

        print("Saved new best model.")
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

print("Best validation Dice at threshold 0.5:", best_val_dice)

In [ ]:
import matplotlib.pyplot as plt

# Load best raw model.
ckpt = torch.load(RUN_DIR / "best_model_raw_threshold.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

def collect_probs_and_targets(loader):
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu().numpy()
            targets = masks.numpy()

            all_probs.append(probs)
            all_targets.append(targets)

    return np.concatenate(all_probs), np.concatenate(all_targets)

probs, targets = collect_probs_and_targets(val_loader)

def dice_iou_np(probs, targets, threshold):
    preds = (probs > threshold).astype(np.uint8)
    targets = (targets > 0.5).astype(np.uint8)

    intersection = (preds & targets).sum(axis=(1,2,3))
    pred_sum = preds.sum(axis=(1,2,3))
    target_sum = targets.sum(axis=(1,2,3))

    smooth = 1e-6
    dice = (2 * intersection + smooth) / (pred_sum + target_sum + smooth)
    iou = (intersection + smooth) / (pred_sum + target_sum - intersection + smooth)

    return float(dice.mean()), float(iou.mean())

thresholds = np.arange(0.20, 0.81, 0.05)
rows = []

for t in thresholds:
    d, i = dice_iou_np(probs, targets, t)
    rows.append({"threshold": float(t), "val_dice": d, "val_iou": i})

threshold_df = pd.DataFrame(rows)
threshold_df.to_csv(RUN_DIR / "threshold_tuning.csv", index=False)
display(threshold_df)

best_row = threshold_df.sort_values("val_dice", ascending=False).iloc[0]
best_threshold = float(best_row["threshold"])

print("Best threshold:", best_threshold)
print("Best tuned val Dice:", best_row["val_dice"])
print("Best tuned val IoU:", best_row["val_iou"])

# Save final tuned checkpoint.
torch.save({
    "model_state_dict": model.state_dict(),
    "img_size": IMG_SIZE,
    "threshold": best_threshold,
    "val_dice": float(best_row["val_dice"]),
    "val_iou": float(best_row["val_iou"]),
    "model_type": "ResNet34UNet_binary_lichen",
}, RUN_DIR / "best_model_tuned.pt")

# Save validation overlays.
OVERLAY_DIR = RUN_DIR / "val_overlays"
OVERLAY_DIR.mkdir(exist_ok=True)

def unnormalize_image(tensor_chw):
    img = tensor_chw.detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = img * IMAGENET_STD + IMAGENET_MEAN
    img = np.clip(img * 255, 0, 255).astype(np.uint8)
    return img

saved = 0
model.eval()

with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(val_loader):
        images_gpu = images.to(device)
        logits = model(images_gpu)
        batch_probs = torch.sigmoid(logits).cpu().numpy()

        for j in range(images.shape[0]):
            img = unnormalize_image(images[j])
            true_mask = masks[j, 0].numpy() > 0.5
            pred_mask = batch_probs[j, 0] > best_threshold

            overlay = img.copy()
            overlay[pred_mask] = [255, 70, 70]      # predicted lichen = red
            overlay[true_mask] = [70, 255, 70]      # true lichen = green

            blended = cv2.addWeighted(img, 0.65, overlay, 0.35, 0)

            cv2.imwrite(
                str(OVERLAY_DIR / f"val_{saved:03d}_overlay_red_pred_green_true.png"),
                cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)
            )
            cv2.imwrite(
                str(OVERLAY_DIR / f"val_{saved:03d}_pred_mask.png"),
                (pred_mask.astype(np.uint8) * 255)
            )

            saved += 1

            if saved >= 20:
                break

        if saved >= 20:
            break

print("Saved validation overlays to:", OVERLAY_DIR)
print("Final model saved to:", RUN_DIR / "best_model_tuned.pt")

In [ ]:
NEW_IMAGE_DIR = Path("/content/new_lichen_images")
PRED_DIR = Path("/content/predicted_lichen_masks")
PRED_MASK_DIR = PRED_DIR / "masks"
PRED_OVERLAY_DIR = PRED_DIR / "overlays"

PRED_MASK_DIR.mkdir(parents=True, exist_ok=True)
PRED_OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

# Load tuned model.
ckpt = torch.load(RUN_DIR / "best_model_tuned.pt", map_location=device)
model = ResNet34UNet(pretrained=False).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

IMG_SIZE = int(ckpt["img_size"])
THRESHOLD = float(ckpt["threshold"])

print("Using threshold:", THRESHOLD)

def preprocess_for_prediction(rgb, size):
    resized = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)
    x = resized.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = np.transpose(x, (2, 0, 1))[None, :, :, :].astype(np.float32)
    return torch.tensor(x)

pred_records = []

image_paths = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
    image_paths.extend(list(NEW_IMAGE_DIR.glob(ext)))

image_paths = sorted(image_paths)
print("New images found:", len(image_paths))

with torch.no_grad():
    for img_path in tqdm(image_paths):
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            continue

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]

        x = preprocess_for_prediction(rgb, IMG_SIZE).to(device)
        logits = model(x)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

        pred_small = (prob > THRESHOLD).astype(np.uint8) * 255
        pred_mask = cv2.resize(pred_small, (w, h), interpolation=cv2.INTER_NEAREST)

        mask_out = PRED_MASK_DIR / f"{img_path.stem}_mask.png"
        overlay_out = PRED_OVERLAY_DIR / f"{img_path.stem}_overlay.png"

        cv2.imwrite(str(mask_out), pred_mask)

        overlay = rgb.copy()
        overlay[pred_mask > 0] = [255, 70, 70]
        blended = cv2.addWeighted(rgb, 0.68, overlay, 0.32, 0)
        cv2.imwrite(str(overlay_out), cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))

        coverage = float((pred_mask > 0).mean() * 100)
        pred_records.append({
            "image": img_path.name,
            "mask_path": str(mask_out),
            "overlay_path": str(overlay_out),
            "lichen_coverage_percent": coverage,
        })

pred_df = pd.DataFrame(pred_records)
pred_df.to_csv(PRED_DIR / "predicted_coverage.csv", index=False)

print("Saved masks to:", PRED_MASK_DIR)
print("Saved overlays to:", PRED_OVERLAY_DIR)
display(pred_df.head())

In [ ]:
import shutil

zip_path = "/content/lichen_binary_100_prototype_outputs.zip"

if Path(zip_path).exists():
    Path(zip_path).unlink()

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content",
    base_dir="runs/lichen_binary_resnet34_unet_100"
)

print("Created:", zip_path)

In [ ]:
# ================================
# LichenLens Visual Aid Stats + Figures
# Run after training + threshold tuning
# ================================

import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# ----------------
# Paths
# ----------------

RUN_DIR = Path("/content/runs/lichen_binary_resnet34_unet_100")
EXPORT_DIR = RUN_DIR / "visual_aid_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LOG_PATH = RUN_DIR / "training_log.csv"
THRESHOLD_PATH = RUN_DIR / "threshold_tuning.csv"
TUNED_MODEL_PATH = RUN_DIR / "best_model_tuned.pt"
TRAIN_SPLIT_PATH = RUN_DIR / "train_split.csv"
VAL_SPLIT_PATH = RUN_DIR / "val_split.csv"

# ----------------
# Safety checks
# ----------------

if "model" not in globals():
    raise RuntimeError("Run this in the same notebook after training, so `model` exists.")

if "val_loader" not in globals():
    raise RuntimeError("Run this in the same notebook after creating `val_loader`.")

if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best tuned checkpoint if available
if TUNED_MODEL_PATH.exists():
    ckpt = torch.load(TUNED_MODEL_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    best_threshold = float(ckpt.get("threshold", 0.5))
else:
    print("WARNING: best_model_tuned.pt not found. Using threshold = 0.5.")
    best_threshold = 0.5

model = model.to(device)
model.eval()

# ImageNet values from the training code
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def unnormalize_image_tensor(tensor_chw):
    """Convert normalized CHW tensor back to RGB uint8."""
    img = tensor_chw.detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = img * IMAGENET_STD + IMAGENET_MEAN
    img = np.clip(img * 255, 0, 255).astype(np.uint8)
    return img

def safe_divide(a, b):
    return float(a / b) if b != 0 else 0.0

def compute_binary_metrics(pred, target):
    """pred and target are boolean masks."""
    pred = pred.astype(bool)
    target = target.astype(bool)

    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, ~target).sum()
    fn = np.logical_and(~pred, target).sum()
    tn = np.logical_and(~pred, ~target).sum()

    dice = safe_divide(2 * tp, 2 * tp + fp + fn)
    iou = safe_divide(tp, tp + fp + fn)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    pixel_accuracy = safe_divide(tp + tn, tp + fp + fn + tn)

    pred_coverage = pred.mean() * 100
    true_coverage = target.mean() * 100
    coverage_abs_error = abs(pred_coverage - true_coverage)

    return {
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "pixel_accuracy": pixel_accuracy,
        "pred_coverage_percent": pred_coverage,
        "true_coverage_percent": true_coverage,
        "coverage_abs_error_percent": coverage_abs_error,
    }

def make_error_overlay(rgb, pred, target):
    """
    Green = correct lichen prediction
    Red = false positive
    Blue = missed lichen
    """
    pred = pred.astype(bool)
    target = target.astype(bool)

    tp = np.logical_and(pred, target)
    fp = np.logical_and(pred, ~target)
    fn = np.logical_and(~pred, target)

    overlay = rgb.copy()

    overlay[tp] = [0, 220, 80]       # correct lichen
    overlay[fp] = [255, 70, 70]      # false positive
    overlay[fn] = [70, 140, 255]     # missed lichen

    blended = cv2.addWeighted(rgb, 0.58, overlay, 0.42, 0)
    return blended

# ----------------
# Evaluate validation set
# ----------------

all_rows = []
sample_store = []

sample_index = 0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)

        preds = (probs > best_threshold).detach().cpu().numpy()
        targets = (masks > 0.5).detach().cpu().numpy()

        for i in range(images.shape[0]):
            pred = preds[i, 0].astype(bool)
            target = targets[i, 0].astype(bool)

            metrics = compute_binary_metrics(pred, target)
            metrics["sample_index"] = sample_index

            all_rows.append(metrics)

            rgb = unnormalize_image_tensor(images[i])
            error_overlay = make_error_overlay(rgb, pred, target)

            sample_store.append({
                "sample_index": sample_index,
                "rgb": rgb,
                "pred": pred,
                "target": target,
                "error_overlay": error_overlay,
                "metrics": metrics,
            })

            sample_index += 1

val_stats = pd.DataFrame(all_rows)
val_stats.to_csv(EXPORT_DIR / "validation_sample_metrics.csv", index=False)

# ----------------
# Summary stats for caption/table
# ----------------

n_train = len(pd.read_csv(TRAIN_SPLIT_PATH)) if TRAIN_SPLIT_PATH.exists() else None
n_val = len(val_stats)

summary = {
    "train_images": n_train,
    "validation_images": n_val,
    "best_threshold": best_threshold,
    "mean_dice": val_stats["dice"].mean(),
    "median_dice": val_stats["dice"].median(),
    "mean_iou": val_stats["iou"].mean(),
    "median_iou": val_stats["iou"].median(),
    "mean_precision": val_stats["precision"].mean(),
    "mean_recall": val_stats["recall"].mean(),
    "mean_pixel_accuracy": val_stats["pixel_accuracy"].mean(),
    "mean_coverage_abs_error_percent": val_stats["coverage_abs_error_percent"].mean(),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(EXPORT_DIR / "visual_aid_model_summary.csv", index=False)

print("===== MODEL SUMMARY =====")
display(summary_df)
print("Saved stats to:", EXPORT_DIR)

In [ ]:
# ================================
# Visual 1: Model Performance Dashboard
# ================================

log_df = pd.read_csv(TRAIN_LOG_PATH) if TRAIN_LOG_PATH.exists() else None
threshold_df = pd.read_csv(THRESHOLD_PATH) if THRESHOLD_PATH.exists() else None

mean_dice = summary["mean_dice"]
mean_iou = summary["mean_iou"]
mean_precision = summary["mean_precision"]
mean_recall = summary["mean_recall"]
coverage_mae = summary["mean_coverage_abs_error_percent"]

fig = plt.figure(figsize=(13, 7.5), dpi=220)
gs = fig.add_gridspec(
    nrows=3,
    ncols=6,
    height_ratios=[0.85, 1.35, 1.35],
    hspace=0.65,
    wspace=0.45
)

fig.suptitle(
    "LichenLens Binary Segmentation: Validation Performance",
    fontsize=19,
    fontweight="bold",
    y=0.98
)

def metric_card(ax, title, value, subtitle=""):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("#F4F7FA")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.text(
        0.5, 0.65, value,
        ha="center", va="center",
        fontsize=19,
        fontweight="bold"
    )
    ax.text(
        0.5, 0.34, title,
        ha="center", va="center",
        fontsize=9.5,
        fontweight="bold"
    )
    if subtitle:
        ax.text(
            0.5, 0.14, subtitle,
            ha="center", va="center",
            fontsize=7.5
        )

cards = [
    ("Dice / F1", f"{mean_dice:.3f}", "main segmentation score"),
    ("IoU", f"{mean_iou:.3f}", "overlap quality"),
    ("Precision", f"{mean_precision:.3f}", "false positives controlled"),
    ("Recall", f"{mean_recall:.3f}", "missed lichen controlled"),
    ("Coverage Error", f"{coverage_mae:.1f}%", "mean absolute error"),
    ("Val. Images", f"{n_val}", f"threshold={best_threshold:.2f}"),
]

for i, (title, value, subtitle) in enumerate(cards):
    ax = fig.add_subplot(gs[0, i])
    metric_card(ax, title, value, subtitle)

# Learning curve
ax1 = fig.add_subplot(gs[1:, :3])

if log_df is not None and {"epoch", "val_dice", "val_iou"}.issubset(log_df.columns):
    ax1.plot(log_df["epoch"], log_df["val_dice"], linewidth=2.5, label="Validation Dice")
    ax1.plot(log_df["epoch"], log_df["val_iou"], linewidth=2.5, label="Validation IoU")
    ax1.set_title("Learning Curve", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Score")
    ax1.set_ylim(0, 1.02)
    ax1.grid(alpha=0.25)
    ax1.legend(frameon=False, fontsize=9)
else:
    ax1.text(0.5, 0.5, "training_log.csv not found", ha="center", va="center")
    ax1.axis("off")

# Threshold tuning curve
ax2 = fig.add_subplot(gs[1:, 3:])

if threshold_df is not None and {"threshold", "val_dice", "val_iou"}.issubset(threshold_df.columns):
    ax2.plot(threshold_df["threshold"], threshold_df["val_dice"], linewidth=2.5, label="Dice")
    ax2.plot(threshold_df["threshold"], threshold_df["val_iou"], linewidth=2.5, label="IoU")
    ax2.axvline(best_threshold, linestyle="--", linewidth=2, alpha=0.75)

    ax2.set_title("Decision Threshold Tuning", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Mask Probability Threshold")
    ax2.set_ylabel("Validation Score")
    ax2.set_ylim(0, 1.02)
    ax2.grid(alpha=0.25)
    ax2.legend(frameon=False, fontsize=9)

    ax2.text(
        best_threshold,
        0.08,
        f"best = {best_threshold:.2f}",
        rotation=90,
        va="bottom",
        ha="right",
        fontsize=9
    )
else:
    ax2.text(0.5, 0.5, "threshold_tuning.csv not found", ha="center", va="center")
    ax2.axis("off")

fig.text(
    0.5, 0.015,
    "Caption: Binary U-Net evaluated against manually reviewed lichen masks. Dice/IoU measure mask overlap; coverage error shows effect on lichen coverage estimation.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

out_png = EXPORT_DIR / "visual_1_model_performance_dashboard.png"
out_pdf = EXPORT_DIR / "visual_1_model_performance_dashboard.pdf"

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()

print("Saved:", out_png)
print("Saved:", out_pdf)

In [ ]:
# ================================
# Visual 2: Human Mask vs Model Prediction Examples
# ================================

# Pick worst, median, and best examples by Dice.
val_stats_sorted = val_stats.sort_values("dice").reset_index(drop=True)

pick_rows = []

if len(val_stats_sorted) >= 3:
    pick_rows = [
        val_stats_sorted.iloc[0],                       # worst
        val_stats_sorted.iloc[len(val_stats_sorted)//2], # median
        val_stats_sorted.iloc[-1],                      # best
    ]
else:
    pick_rows = [val_stats_sorted.iloc[i] for i in range(len(val_stats_sorted))]

picked_indices = [int(r["sample_index"]) for r in pick_rows]

sample_by_idx = {s["sample_index"]: s for s in sample_store}
picked_samples = [sample_by_idx[idx] for idx in picked_indices]

fig, axes = plt.subplots(
    1,
    len(picked_samples),
    figsize=(13, 4.6),
    dpi=240
)

if len(picked_samples) == 1:
    axes = [axes]

fig.suptitle(
    "Mask Validation Examples: Model Prediction vs Human-Corrected Mask",
    fontsize=17,
    fontweight="bold",
    y=0.98
)

for ax, sample in zip(axes, picked_samples):
    m = sample["metrics"]
    overlay = sample["error_overlay"]

    ax.imshow(overlay)
    ax.axis("off")

    ax.set_title(
        f"Dice={m['dice']:.2f} | IoU={m['iou']:.2f}\nCoverage error={m['coverage_abs_error_percent']:.1f}%",
        fontsize=10,
        fontweight="bold"
    )

fig.text(
    0.5,
    0.06,
    "Green = correct lichen prediction     Red = false positive     Blue = missed lichen",
    ha="center",
    fontsize=11,
    fontweight="bold"
)

fig.text(
    0.5,
    0.015,
    "Caption: Representative validation images show where the model agrees with human-corrected binary masks and where it still requires improvement.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.08, 1, 0.92])

out_png = EXPORT_DIR / "visual_2_mask_validation_examples.png"
out_pdf = EXPORT_DIR / "visual_2_mask_validation_examples.pdf"

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()

print("Saved:", out_png)
print("Saved:", out_pdf)